<a href="https://colab.research.google.com/github/Skateboardguy/Python_Finance_data_workshop/blob/main/us_fin_refine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 金融數據工作坊 - 爬蟲


## 數據專案思維

- **核心目標:** 想解決什麼問題？(例如：增加營收、降低成本、服務穩定)
- **數據需求:** 需要哪些維度的資料？時效性為何？(即時 or 每日)
- **取得渠道:**
   - 內部數據: 公司資料庫、日誌記錄 (Logs)
   - 外部數據: 官方網站、政府公開資訊、第三方整理、社群媒體


## 金融數據工作坊系列
- **核心目標**: 想知道某幾檔股票過去幾年的報酬率
- **數據需求**: 日期、每日收盤後價格、成交量
- **取得渠道**: 外部網站(官方網站、第三方整理)

因為我們本身不是交易所，也不是券商，沒有內部交易資料庫，資料來源只能是外部網站。  
如何能有效率、自動化地收集這些資料，我們就得藉助網路爬蟲：  

## 網路爬蟲
網路就像一張巨大的蜘蛛網，每一個節點就像是頁面，每一條線則是連結。  
我們平常在瀏覽網站的時候，可以從一個頁面，利用連結再去訪問下一個頁面，  
而當我們發現網路上的內容對自己有幫助時，通常會將它保存下來。  
用程式自動地去逐一訪問所有頁面(Crawling)與擷取所需要的資料(Parsing)，這就是網路爬蟲(Web Scraping)。


<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/network.jpg" width="50%"/>

而要讓網路爬蟲能正確抓取我們想要的資料，需要先理解網頁運作的原理。

# 網頁運作原理


## 以 Chrome 為觀察工具

### 開發人員工具
- 對網頁內容按右鍵 > 檢查
- 在右方選單 > 更多工具 > 開啟『開發人員工具』





<table>
<tr>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-dev1.png"/>
</td>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-dev2.png"/>
</td>
</tr>
</table>


#### Network
- 可以觀察網頁上互動背後的運作原理
- 可以用 Clear 清空之前的網頁互動背後的記錄，然後再將網頁重新整理
- 可利用搜尋功能找出想要的資料是如何取得

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-response.png)

#### Element
- 可以知道 Chrome 將各種元素翻譯後的結果

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-elements.png)

### 檢視網頁原始碼
- 對網頁內容按右鍵 > 檢視網頁原始碼
- 網址前面加上 `view-source:`

<table>
<tr>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-raw.png"/>
</td>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-view_source.png"/>
</td>
</tr>
</table>


想像我們平常看到的圖文並茂的網頁畫面是料理，  
Element 呈現的是加工後(清洗、切製、烹調後)的食材，  
而網頁原始碼就是食材最基礎的樣貌。

接下來我們會以 輝達 NVIDIA 為範例，   
在不同的金融網頁中，利用 Chrome 觀察並了解網頁運作原理


![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-header.png)

## HTTP/HTTPS 協定
HTTP(HyperText Transfer Protocol) / HTTPS(HTTP Secure)   
是一個客戶端(瀏覽器)和伺服器端(網站)之間請求(Request)和回應(Response)的標準

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-http.png)

## GET
HTTP/HTTPS Request 的方法之一，很簡單方便
- 更改網址，就可以得到不同的頁面資訊
- 像寄明信片，所有資訊都在上面

e.g.
- https://tw.stock.yahoo.com/quote/NVDA
- https://tw.stock.yahoo.com/quote/AAPL

<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/postcard.jpg" width="50%"/>

### 網頁組成元素
- HTML：網頁內容骨架的標記型語言
- CSS：裝飾網頁內容的樣式表語言
- Javascript(JS)：負責網頁互動的程式語言  
  
> 有些內容是直接以 HTML 呈現，有些是利用 Javascript 組合而成  
> 在開發人員工具使用 Ctrl/Cmd + Shift + P 可以「Disable Javascript」，網頁重整後 Javascript 產生的內容就會消失。

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-network.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-js.png)

https://tw.stock.yahoo.com/quote/NVDA 的例子中，股價圖就是由 Javascript 產生的。

這種網頁實作方式是後端單純將資料傳出，前端再負責呈現畫面，  
因此找到後端跟前端溝通的網址，就可以只要處理資料，  
不需要理會網頁畫面看到的圖片配色版型...etc.
這個網址我們常稱為「API」。

### [練習] 找尋 API 網址
但實際上我們所需要的並不是圖本身，而是圖上的數字，  
因為我們需要的是「每日股價」，請先將股價圖切換至「全部」，    
再利用開發人員工具搜尋圖上的數字，來找到資料對應的網址

> 不建議拿最新日期數字來查詢，因為重複出現機率滿高的

```
https://tw.stock.yahoo.com/_____/___/resource/FinanceChartService.ApacLibraCharts;...symbols=%5B%22NVDA%22%5D...
```

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/yahoo_vis.png)

## Request Header
當客戶端發送一個 HTTP Request 給伺服器時，
除了網址與方法（GET / POST）之外，  
還會附上一份「說明文件」，這份說明文件就是 Request Header，  
伺服器會根據這些資訊，決定怎麼回應你。

可以把它想像成：寄件時，會需要提供寄件人、寄件地址、內容物哪面朝上...等資訊



In [ ]:
import requests
url = "https://tw.stock.yahoo.com/quote/NVDA"
headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
response = requests.get(url, headers=headers)

with open("nvda_yahoo.html","w") as fo:
    fo.write(response.text)

可以在 colab 左側的「資料夾」圖示、按下重新整理，  
即可找到剛用爬蟲下載的網頁原始碼，再對該檔案下載後，  
使用瀏覽器開啟即可看到爬蟲下載的網頁預覽。

<table>
<tr>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/colab_file.png"/>
</td>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/colab_file_download.png"/>
</td>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/colab_file_download_view.png"/>
</td>
</tr>
</table>


## POST
HTTP/HTTPS Request 的方法之一
- 同一個網址，透過更改內容物 Request Payload (Request Body) 得到不同的頁面資訊
- 常用於登入或送出表單的情境
- 像寄有信封的郵件，可以裝更多資料、  
郵差只會知道收寄件人、不會看到內容物


<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/envelope.jpg" width="50%"/>

以 https://statementdog.com/analysis/NVDA 為例，  
其中部分的資訊是以 https://statementdog.com/api/v2/indicators 用不同的 Request Payload 獲得。

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-post.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/chrome-post_body.png)

In [2]:
import requests
import json

url = "https://statementdog.com/api/v2/indicators"

payload = {
    "series_keys":[
        {
            "series_class":"Common::LatestDatum",
            "ticker":"NVDA"
        }
    ],
    "date_range":{
        "start":"2025-02-21T16:26:59.214Z",
        "end":"2026-02-21T16:26:59.215Z"
    }
}
headers = {
  'Content-Type': 'application/json'
}

response = requests.post(url, headers=headers, data=json.dumps(payload))
with open("nvda_statementdog.html","w") as fo:
    fo.write(response.text)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# 實戰要點
- 理解背後原理：看懂資料來源
- 觀察異同之處：網址或 Request Payload 的資料變換
- 小量實作試驗，再逐步擴大
- Crawling 與 Parsing 分開處理
    - 遇到網路不穩定、被封鎖：可以從上次的斷點繼續 Crawling，不需從頭開始
    - 遇到網站改版：擷取失敗只需要重新撰寫新的 Parsing 邏輯，不需要再重新 Crawling

## 網路爬蟲的倫理素養
- 尊重網站的 robots.txt 規範
- 控制抓取頻率，避免對伺服器造成負擔
- 留意資料用途是否符合網站的使用條款

> 若僅作為學習與個人分析用途，風險通常較低；但若涉及公開散佈或商業使用，則應特別留意相關規範。

### 延伸閱讀
- [AI橫行，30年前寫給「君子」的robots.txt擋得住今日的爬蟲巨獸嗎？](https://dq.yam.com/post/16602)


## robots.txt
robots.txt 它代表網站的「意願聲明」，雖然不是法律文件，    
但如果網站明確禁止(Disallow)某些路徑被哪些爬蟲(User-agent)抓取，就應該尊重。
  
### robots.txt 位置
將網頁的「網域名稱 (Domain Name)」後加上 `robots.txt`，  
前述程式碼範例中使用的網站之 robots.txt 都沒有被列在 Disallow 中：
- https://tw.stock.yahoo.com/robots.txt
    > https://tw.stock.yahoo.com/quote/NVDA
- https://statementdog.com/robots.txt
    > https://statementdog.com/api/v2/indicators

但在前述 [[練習] 找尋 API 網址的答案](https://colab.research.google.com/drive/1f9v68kxH6ZdATPFQAJaTc6SBuj0JzILO#scrollTo=DJLhz4TlR30K&line=10&uniqifier=1)，在 [robots.txt](https://tw.stock.yahoo.com/robots.txt) 是 Disallow

## 資料來源與驗證
官方資料通常最具權威性，但必須理解不同市場制度的差異。  
台灣股票集中於臺灣證券交易所，而美國則分屬不同交易市場，    
例如 納斯達克證券市場 NASDAQ、紐約證券交易所 NYSE、美國證券交易所 AMEX ...。

然而，官方來源的資料未必容易直接取得，可能存在存取限制、API 規範或商業授權條件。  
在實務上，透過交叉驗證確保數據正確性，也可以使用第三方資料服務作為替代方案。

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/data_valid_yahoo.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/data_valid_stooq.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/data_valid_nasdaq.png)

# Crawling

## [stooq](https://stooq.com/)
- 無限制的 [robots.txt](https://stooq.com/robots.txt)
- 資料非常好抓取
- 有提供 csv 下載
- 有可能會被網站暫時封鎖


> *Redistribution of data found on the website is not allowed without the consent of Stooq.*

### 抓取網頁內容

先測試爬蟲是否抓的到網頁內容

In [5]:
import requests
url = "https://stooq.com/q/d/?s=nvda.us"
response = requests.get(url)

with open("nvda_stooq.html","w") as fo:
    fo.write(response.text)

如果爬蟲下載的網頁無法順利用瀏覽器開啟，不一定是下載失敗  
可以檢查網頁上想要的資訊是否有在網頁原始碼中
> - 建議拿小數點後有三碼的數字，重複率較低
> - 不建議拿最新日期數字來查詢，因為重複出現機率滿高的
> - 不建議拿成交量，因為千位數逗號不見得是原始資料

In [6]:
with open("nvda_stooq.html") as fi:
    html = fi.read()
    print("某日價格" in html)

False


我們要抓的歷史資料，在網頁上會分成很多頁，可以點進不同頁看網址的變化
- 找到網址規律後，第一頁也可以用同樣規律抓嗎？
- 超出頁數的網頁會顯示什麼？

#### [練習] 可抓取多頁的爬蟲
如果我們要抓取所有每日歷史資料，只抓第一頁是不夠的，  
可以利用 for 迴圈，配合字串格式化替換網址頁數，就可以抓取多頁的網頁內容。

##### 實戰要點
- 很多頁的資料，可以建立一個專屬的資料夾存放。
- 需要設定兩次間隔抓取的時間，降低被網站封鎖的風險

In [ ]:
import requests
import os
import random
import time


dir_name = "nvda_stooq"

### 建立資料夾
if dir_name not in os.listdir():
    os.makedirs(dir_name)

for page in range(1,4): # will crawling page 1,2,3
    url = f"https://stooq.com/q/d/_________{page}"
    response = requests.get(url)

    with open(f"{dir_name}/{page}.html","w") as fo:
        fo.write(response.text)
    time.sleep(random.randint(3,7)) # 程式暫停 3~7秒

In [ ]:
# 檢查網頁上想要的資訊是否有在網頁原始碼中
with open("nvda_stooq/1.html") as fi:
    html = fi.read()
    print("某日價格" in html)

### 下載檔案

In [ ]:
import requests
url = "http://stooq.com/q/d/l/?s=nvda.us&i=d"
response = requests.get(url)

with open("nvda_stooq.csv","wb") as fo:
    fo.write(response.content)

相比前面抓取網頁內容
- 需要抓取多頁，對網站負擔較高，也增加被網站封鎖的風險
- 需要解析網頁原始碼

下載檔案相對是一個更為方便的選擇

## [NASDAQ](https://www.nasdaq.com/)
- 美國最大的證券交易市場
- [robots.txt](https://www.nasdaq.com/robots.txt)

先測試爬蟲是否抓的到網頁內容

In [7]:
import requests
url = "https://www.nasdaq.com/market-activity/stocks/nvda/historical"
headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
response = requests.get(url, headers=headers)

with open("nvda_nasdaq.html","w") as fo:
    fo.write(response.text)

### [練習] 找尋 API 網址
如果要抓所有的歷史資料，要記得將圖表切到「MAX」，  
開啟開發人員工具的 Network 分頁，然後重新整理網頁，  
再利用開發人員工具搜尋圖上的數字，來找到資料對應的網址

> - 建議拿小數點後有三碼的數字，重複率較低
> - 不建議拿最新日期數字來查詢，因為重複出現機率滿高的
> - 不建議拿成交量，因為千位數逗號不見得是原始資料

https://www.nasdaq.com/market-activity/stocks/nvda/historical?page=1&rows_per_page=10&timeline=y10

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/nasdaq_nvda.png)

找到網址規律後，
- 如何調整查詢日期？
- 如何顯示更多筆的資料？

會發覺跟 https://www.nasdaq.com/market-activity/stocks/nvda/historical?page=1&rows_per_page=10&timeline=y10
的域名不同，  
所以需要再檢查一下 [robots.txt](https://api.nasdaq.com/robots.txt)

## [Financial Modeling Prep](https://site.financialmodelingprep.com/)
- 和 [NASDAQ](https://data.nasdaq.com/market-data-vendors) 有數據合作
- 有 SOC 2 Type 1 保證，資料來源可靠
- 有 [API 文件](https://site.financialmodelingprep.com/developer/docs) 可查詢如何使用
- 需要[申請拿取 API Key](https://site.financialmodelingprep.com/register)
- 不需考慮 robots.txt
- 不會有被封鎖的風險
- 免費版可拿到五年的歷史資料，一天可以使用 250 次

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/fmp.png)

<table>
<tr>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/data_valid_fmp.png"/>
</td>
<td>
<img src="https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/data_valid_nasdaq-2.png"/>
</td>
</tr>
</table>


跟 NASDAQ 比較成交量差異更小，但開盤價等資訊只有小數點後兩位數

In [ ]:
import requests
API_KEY = "________" # 請填上 Financial Modeling Prep 的 API Key
url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol=NVDA&apikey={API_KEY}"
response = requests.get(url)

with open("nvda_fmp.html","w") as fo:
    fo.write(response.text)

# Parsing

## HTML

### Python 語法
可利用字串、串列語法取得想要的資訊 e.g.
- `split`
- `join`
- `append`

In [ ]:
with open("nvda_yahoo.html") as fi:
    html = fi.read()
chart_data = html.split('"timestamp":')[-1].split(',"comparisons"')[0]
json_style_data = '{"timestamp":'+chart_data+'}'
print(json_style_data)

### [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
專門處理 HTML/XML 格式的 Python 工具

開發人員工具的 Element 是 Chrome 將各種元素翻譯後的結果，  
不一定跟網頁原始碼完全相同，但網頁原始碼有時格式很亂，  
可以用 `prettify()` 觀察排版後真正的網頁原始碼。


In [8]:
from bs4 import BeautifulSoup

with open("nvda_stooq.html") as fi:
    html = fi.read()

soup = BeautifulSoup(html, "html.parser")
print(soup.prettify())

<meta charset="utf-8"/>
<title>
 Stooq
</title>
<center style="font-family:arial;margin-top:50px">
 <p>
  <a href="/">
   <img height="68" src="//static.stooq.com/stooq.svg"/>
  </a>
  <p style="font-size:x-large">
   The page you requested does not exist
   <br/>
   or has been moved
   <p>
    <small>
     <a href="/">
      Main Page
     </a>
    </small>
   </p>
  </p>
 </p>
</center>



![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-html.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-bs4-01.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-bs4-02.png)

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-bs4-02.png)

In [ ]:
from bs4 import BeautifulSoup

with open("nvda_stooq.html") as fi:
    html = fi.read()

soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", attrs={"class":"fth1"}) # find:     只找一個
rows = table.find_all("tr")                        # find_all: 找全部

for row in rows:
    cols = row.find_all("td")
    data = []
    data = [col.get_text() for col in cols]
    print(data)

## [JSON](https://docs.python.org/zh-tw/3/library/json.html)
- `json.loads(字串)`：json 格式字串 -> python 資料型別(串列, 字典, 字串, 數值)
- `json.dumps(資料型別)`：python 資料型別 -> json格式字串

> 輔助工具：[JSON Structure Explorer in Python](https://python-practical-workshop.github.io/json-parser/)


In [13]:
import json

with open("nvda_yahoo.html") as fi:
    html = fi.read()
chart_data = html.split('"timestamp":')[-1].split(',"comparisons"')[0]
json_style_data = '{"timestamp":'+chart_data+'}'
print(json_style_data)

json_data = json.loads(json_style_data)

timestamps = json_data["timestamp"]
opens = json_data["indicators"]["quote"][0]["open"]
highs = json_data["indicators"]["quote"][0]["high"]
lows = json_data["indicators"]["quote"][0]["low"]
closes = json_data["indicators"]["quote"][0]["close"]
volumes = json_data["indicators"]["quote"][0]["volume"]
data = list(zip(timestamps, opens, highs, lows, closes, volumes))

print(timestamps[1], opens[1], highs[1], lows[1], closes[1], volumes[1])
print(data[1])

{"timestamp":[1782394200,1782394260,1782394320,1782394380,1782394440,1782394500,1782394560,1782394620,1782394680,1782394740,1782394800,1782394860,1782394920,1782394980,1782395040,1782395100,1782395160,1782395220,1782395280,1782395340,1782395400,1782395460,1782395520,1782395580,1782395640,1782395700,1782395760,1782395820,1782395880,1782395940,1782396000,1782396060,1782396120,1782396180,1782396240,1782396300,1782396360,1782396420,1782396480,1782396540,1782396600,1782396660,1782396720,1782396780,1782396840,1782396900,1782396960,1782397020,1782397080,1782397140,1782397200,1782397260,1782397320,1782397380,1782397440,1782397500,1782397560,1782397620,1782397680,1782397740,1782397800,1782397860,1782397920,1782397980,1782398040,1782398100,1782398160,1782398220,1782398280,1782398340,1782398400,1782398460,1782398520,1782398580,1782398640,1782398700,1782398760,1782398820,1782398880,1782398940,1782399000,1782399060,1782399120,1782399180,1782399240,1782399300,1782399360,1782399420,1782399480,1782399

![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/yahoo_json.png)

### [練習] 解析 JSON

In [14]:
from datetime import datetime
import json

with open("nvda_fmp.html") as fi:
    html = fi.read()

json_data = json.loads(html)
dates = []
opens = []
highs = []
lows = []
closes = []
volumes = []
for row in json_data:
    dates.append(row["date"])
    opens.append(row["____"])
    highs.append(row["____"])
    lows.append(row["____"])
    closes.append(row["____"])
    volumes.append(row["volume"])

data = list(zip(dates, opens, highs, lows, closes, volumes))
print(data[0])

FileNotFoundError: [Errno 2] No such file or directory: 'nvda_fmp.html'

# 整合實作

## 函式
``` python
def 函式名稱(傳入的參數1, 傳入的參數2, ...):
    處理資料
    return 處理過後的資料
```

- 將會改動的地方設計成傳入的參數
- 如果我們希望被函式處理過的資料，能在後續其他地方使用，就會需要用到 `return`



![](https://raw.githubusercontent.com/Python-Practical-Workshop/crawler/refs/heads/master/imgs/slide-func.png)

越高級的洗衣機，「傳入的參數」可能越多：
- 洗衣類別
- 洗衣時長
- 脫水轉速
- 脫水時長

``` python
洗衣(衣服, 洗衣時長=快速)
```

In [15]:
import json
result = json.loads('{"key":"value"}')
print(type(json.loads))
print(result)
print(type(result))

<class 'function'>
{'key': 'value'}
<class 'dict'>


In [16]:
result = print("Hello")
print(type(print))
print(result)
print(type(result))

Hello
<class 'builtin_function_or_method'>
None
<class 'NoneType'>


## csv 輸出

In [ ]:
import csv

data = [
    ["Date","Open","High","Low","Close","Volume"],
    ["2026-02-20", "186.57", "190.33", "185.9378", "189.82", "178,422,300"],
    ["2026-02-19", "187.055", "188.43", "185.66", "187.90", "126,554,500"],
    ["2026-02-18", "188.75", "190.37", "186.76", "187.98", "164,749,100"],
]

with open("sample.csv", "w") as fo:
    writer = csv.writer(fo)
    writer.writerows(data)

### 改寫成函式
每次處理會更改的有
- 寫入的內容 => 命名為 `data`
- 檔名 => 命名為 `filename`

In [ ]:
import csv
def save_csv(data, filename):
    with open(filename, "w") as fo:
        writer = csv.writer(fo)
        writer.writerows(data)

data = [
    ["Date","Open","High","Low","Close","Volume"],
    ["2026-02-20", "186.57", "190.33", "185.9378", "189.82", "178,422,300"],
    ["2026-02-19", "187.055", "188.43", "185.66", "187.90", "126,554,500"],
    ["2026-02-18", "188.75", "190.37", "186.76", "187.98", "164,749,100"],
]

save_csv(data, "sample_func.csv")


## [練習] Crawling 函式
請將原來的程式碼改寫成函式
``` python
import requests
API_KEY = "________" # 請填上 Financial Modeling Prep 的 API Key
url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol=NVDA&apikey={API_KEY}"
response = requests.get(url)

with open("nvda_fmp.html","w") as fo:
    fo.write(response.text)
```

每次處理會更改的有
- url
- 檔名

如果把 NVDA 換成查詢 GOOGL 會是要改哪些地方？

In [ ]:
import requests
FILENAME_SUFFIX = "_fmp"
API_KEY = "________" # 請填上 Financial Modeling Prep 的 API Key

def crawling(stock_id):
    # url 為此題練習的地方
    url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={____}&apikey={API_KEY}"
    headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
    response = requests.get(url, headers=headers)
    with open(f"{____}{FILENAME_SUFFIX}.html","w") as fo: # 檔名為 此題練習的地方
        fo.write(response.text)

crawling("NVDA")
crawling("GOOGL")
crawling("AAPL")

## Parsing 函式改寫

每次處理會更改的有
- 檔名

In [ ]:
from datetime import datetime
import json

FILENAME_SUFFIX = "_fmp"

def parsing(stock_id):
    with open(f"{stock_id}{FILENAME_SUFFIX}.html") as fi:
        html = fi.read()

    json_data = json.loads(html)
    dates = []
    opens = []
    highs = []
    lows = []
    closes = []
    volumes = []
    for row in json_data:
        dates.append(row["date"])
        opens.append(row["____"])
        highs.append(row["____"])
        lows.append(row["____"])
        closes.append(row["____"])
        volumes.append(row["volume"])

    data = list(zip(dates, opens, highs, lows, closes, volumes))
    print(data[0])

parsing("NVDA")
parsing("GOOGL")
parsing("AAPL")

## 合併所有函式

In [ ]:
import csv
import json
import requests
import time

FILENAME_SUFFIX = "_fmp"
API_KEY = "________" # 請填上 Financial Modeling Prep 申請的 API Key

def save_csv(data, filename):
    with open(filename, "w") as fo:
        writer = csv.writer(fo)
        writer.writerows(data)

def crawling(stock_id):
    # 上述練習的程式碼
    # ......
    # ......
    # ......

def parsing(stock_id):
    # 上述的
    # ......
    # ......
    # ......
    csv_header = ["Date","Open","High","Low","Close","Volume"]
    data  = [csv_header] + data
    save_csv(data, f"{stock_id}{FILENAME_SUFFIX}.csv")


stock_ids = ["NVDA", "GOOGL", "AAPL"]

for stock_id in stock_ids:
    crawling(stock_id)

for stock_id in stock_ids:
    parsing(stock_id)

# Roadmap
以解決問題為目標學習

## 扎實的基礎

### Python 觀念
- [Python 入門](https://python-tutorial-workshop.github.io/#curriculum)
- [浮點數誤差 與 `round()` 的行為差異](https://hackmd.io/@meebox/SkBMIByJq)
- 字串處理與編碼（UTF-8）
- 例外處理（try / except）

In [ ]:
print(0.1+0.2)
print(round(4.5))
print(round(5.5))

### HTTP 與網路機制
- Cookie
- Session
- 狀態碼

## 開發工具與環境
- 加速 Crawling
    - multiprocessing
    - asyncio
    - scrapy
    - Message Queue

- HTML 進階解析
    - XPath (lxml)
    - crawl4AI

- 模擬瀏覽器
    - selenium
    - Playwright

- 開發環境管理
    - Python 版本管理 (pyenv)
    - Python Package Index (PyPI)
    - 虛擬環境 (venv)
    - uv
    - IP 管理
    - 排程工具 (cron)